In [ ]:
import tensorflow as tf

# Check if GPU is available
gpu_available = tf.config.list_physical_devices('GPU')
if gpu_available:
    print("GPU is available and being used:")
    for device in gpu_available:
        print(device)
else:
    print("No GPU available. Using CPU.")

# Optionally, run nvidia-smi to see GPU details (if GPU is available)
!nvidia-smi

In [ ]:
!pip install ultralytics

In [ ]:
!curl -L "https://app.roboflow.com/ds/YOUR_DATASET_ID?key=${ROBOFLOW_API_KEY}" > roboflow.zip; unzip roboflow.zip; rm roboflow.zip


In [ ]:
# Install required packages
!pip install roboflow
!pip install ultralytics

# Step 1: Download and unzip the dataset using direct curl link
!curl -L "https://app.roboflow.com/ds/YOUR_DATASET_ID?key=${ROBOFLOW_API_KEY}" > roboflow.zip; unzip roboflow.zip; rm roboflow.zip
!unzip -o roboflow.zip -d "/content/Mitosis-Detection-YOLO-13"
!rm roboflow.zip

# Step 2: Verify dataset contents
import os
extract_path = "/content/Mitosis-Detection-YOLO-13"
print("Directory contents after extraction:")
print(os.listdir(extract_path))

# Step 3: Create or overwrite the data.yaml file
data_yaml_content = """
train: /content/Mitosis-Detection-YOLO-13/train
val: /content/Mitosis-Detection-YOLO-13/valid
test: /content/Mitosis-Detection-YOLO-13/test
nc: 1
names: ['Mitosis']
"""
with open("/content/Mitosis-Detection-YOLO-13/data.yaml", "w") as f:
    f.write(data_yaml_content)

# Step 4: Create custom YOLOv8 architecture YAML
custom_yaml = """
nc: 1
depth_multiple: 0.67
width_multiple: 0.75

backbone:
  - [-1, 1, Conv, [64, 3, 2]]
  - [-1, 1, Conv, [64, 3, 1]]
  - [-1, 2, C2f, [64, 64, 2, True]]
  - [-1, 1, Conv, [128, 3, 2]]
  - [-1, 1, Conv, [128, 3, 1]]
  - [-1, 4, C2f, [128, 128, 4, True]]
  - [-1, 1, Conv, [256, 3, 2]]
  - [-1, 1, Conv, [256, 3, 1]]
  - [-1, 6, C2f, [256, 256, 6, True]]
  - [-1, 1, Conv, [512, 3, 2]]
  - [-1, 1, Conv, [512, 3, 1]]
  - [-1, 3, C2f, [512, 512, 3, True]]
  - [-1, 1, Conv, [1024, 3, 1]]
  - [-1, 1, Conv, [512, 3, 1]]
  - [-1, 1, Conv, [512, 3, 1]]
  - [-1, 1, SPPF, [512, 5]]

  - [-1, 1, nn.Upsample, [None, 2, 'nearest']]
  - [[-1, 8], 1, Concat, [1]]
  - [-1, 1, C2f, [256, 256, 1, True]]
  - [-1, 1, nn.Upsample, [None, 2, 'nearest']]
  - [[-1, 5], 1, Concat, [1]]
  - [-1, 1, C2f, [128, 128, 1, True]]

head:
  - [-1, 1, Conv, [128, 3, 2]]
  - [[-1, 18], 1, Concat, [1]]
  - [-1, 1, C2f, [256, 256, 1, True]]
  - [-1, 1, Conv, [256, 3, 2]]
  - [[-1, 15], 1, Concat, [1]]
  - [-1, 1, C2f, [512, 512, 1, True]]
  - [[21, 24, 27], 1, Detect, [nc]]
"""

with open("/content/custom_yolov8m.yaml", "w") as f:
    f.write(custom_yaml)



In [ ]:
from ultralytics import YOLO

# Load the YOLOv8n model (can also be yolov8s.pt or your custom .pt file)
model = YOLO("yolov8m.pt")

# Start training
model.train(
    data="/content/Mitosis-Detection-YOLO-13/data.yaml",
    epochs=150,
    imgsz=768,               # 🔍 Higher res helps small objects
    batch=4,
    device=0,
    patience=10,
    optimizer="Adam",
    lr0=0.001,
    degrees=10,
    translate=0.2,
    scale=0.5,
    shear=2.0,
    perspective=0.0005,
    flipud=0.3,
    fliplr=0.5,
    mosaic=1.0,
    close_mosaic=1,         # ✅ Now it's an int
    mixup=0.2,
    copy_paste=0.2,
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,
    cos_lr=True,
    pretrained=True,
    rect=True,              # Helps with varying aspect ratios
    workers=2
)
